# LeetCode 2074 — Reverse Nodes in Even Length Groups bằng Python

Link đề: https://leetcode.com/problems/reverse-nodes-in-even-length-groups/ · Độ khó: **Medium** · Chủ đề: Linked List

Skeleton LeetCode:

```python
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def reverseEvenLengthGroups(self, head: Optional[ListNode]) -> Optional[ListNode]:
        ...
```

## 1. Mục tiêu học

1. Chia danh sách liên kết thành các nhóm có độ dài **dự kiến** `1, 2, 3, …`; nhóm **cuối** có thể ngắn hơn — điều kiện đảo dựa trên độ dài **thực tế**.
2. Đảo một đoạn con của danh sách liên kết **tại chỗ** và nối lại hai đầu (`prev_tail → đầu mới`, `đuôi mới → phần sau`).
3. Quản lý con trỏ "đuôi nhóm trước" để duyệt nhóm kế tiếp — `O(n)` thời gian, `O(1)` bộ nhớ.
4. So sánh với cách đơn giản: chuyển sang mảng, đảo các đoạn, ghi lại giá trị.

## 2. Đề bài & ví dụ

Các nút của danh sách được gán tuần tự vào các nhóm khác rỗng có độ dài theo dãy số tự nhiên `1, 2, 3, 4, …` (nút 1 vào nhóm 1; nút 2–3 vào nhóm 2; nút 4–6 vào nhóm 3; …). Nhóm **cuối** có độ dài **≤** `1 +` độ dài nhóm áp chót (có thể thiếu).

**Đảo** các nút trong mỗi nhóm có **độ dài chẵn**, trả về đầu danh sách.

**Ví dụ 1:** `[5,2,6,3,9,1,7,3,8,4]` → `[5,6,2,3,9,1,4,8,3,7]` (nhóm `[5] [2,6] [3,9,1] [7,3,8,4]`; đảo nhóm 2 và 4).

**Ví dụ 2:** `[1,1,0,6]` → `[1,0,1,6]` (nhóm cuối `[6]` dài 1 — không đảo).

**Ví dụ 3:** `[1,1,0,6,5]` → `[1,0,1,5,6]` (nhóm cuối `[6,5]` dài **2** dù dự kiến là 3 — đảo).

**Ràng buộc:** số nút `[1, 10⁵]`; `0 ≤ val ≤ 10⁵`.

In [1]:
import random
import time
from typing import List, Optional

class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build(vals):
    d = ListNode(); c = d
    for v in vals: c.next = ListNode(v); c = c.next
    return d.next

def to_list(h, limit=10 ** 6):
    out = []
    while h:
        out.append(h.val); h = h.next
        if len(out) > limit: raise RuntimeError("chu trình?")
    return out

examples = [([5, 2, 6, 3, 9, 1, 7, 3, 8, 4], [5, 6, 2, 3, 9, 1, 4, 8, 3, 7]), ([1, 1, 0, 6], [1, 0, 1, 6]), ([1, 1, 0, 6, 5], [1, 0, 1, 5, 6])]
for v, exp in examples:
    print(v, "->", exp)

[5, 2, 6, 3, 9, 1, 7, 3, 8, 4] -> [5, 6, 2, 3, 9, 1, 4, 8, 3, 7]
[1, 1, 0, 6] -> [1, 0, 1, 6]
[1, 1, 0, 6, 5] -> [1, 0, 1, 5, 6]


## 3. Kiến thức nền

### 3.1 Chia nhóm theo dãy `1, 2, 3, …`

Nhóm `k` (đếm từ 1) bắt đầu ở chỉ số `k(k−1)/2` và có độ dài dự kiến `k`; độ dài thực tế là `min(k, n − k(k−1)/2)`.

In [2]:
def groups(n):
    out, start, k = [], 0, 1
    while start < n:
        L = min(k, n - start); out.append((start, L)); start += L; k += 1
    return out

for n in (1, 4, 5, 10):
    print(n, groups(n))

1 [(0, 1)]
4 [(0, 1), (1, 2), (3, 1)]
5 [(0, 1), (1, 2), (3, 2)]
10 [(0, 1), (1, 2), (3, 3), (6, 4)]


### 3.2 Đảo `L` nút bắt đầu từ `prev.next`

Đảo từng nút trong đoạn rồi nối: `prev.next` trỏ vào nút cuối cũ của đoạn (đầu mới), nút đầu cũ (đuôi mới) trỏ vào phần sau đoạn.

In [3]:
def reverse_k(prev, L):
    first = prev.next; cur = first; nxt_prev = None
    for _ in range(L):
        nxt = cur.next; cur.next = nxt_prev; nxt_prev = cur; cur = nxt
    prev.next = nxt_prev            # đầu mới
    first.next = cur                # đuôi mới nối phần sau
    return first                    # đuôi mới (làm prev cho nhóm sau)

d = ListNode(0, build([1, 2, 3, 4, 5]))
tail = reverse_k(d.next, 3)          # đảo 2,3,4
print(to_list(d.next), "| đuôi mới:", tail.val)

[1, 4, 3, 2, 5] | đuôi mới: 2


### 3.3 Dummy node

Nhóm 1 luôn dài 1 (lẻ, không đảo) nên đầu danh sách không bao giờ đổi — nhưng dùng dummy vẫn giúp code đồng nhất.

### 3.4 Đếm độ dài thực tế trước khi quyết định

Phải đi trước tối đa `k` nút để biết nhóm có đủ `k` nút không; tính chẵn lẻ dựa trên số đếm được.

## 4. Ý tưởng nền tảng / trực giác

Duyệt lần lượt từng nhóm `k = 1, 2, 3, …`. Với nhóm `k`: đếm `L` = số nút còn lại tối đa `k`. Nếu `L` chẵn ⇒ đảo `L` nút đó; ngược lại giữ nguyên. Di chuyển con trỏ `prev` tới đuôi của nhóm (sau khi đảo, đuôi là nút **đầu** cũ).

Tính tay **Ví dụ 3** `[1,1,0,6,5]`:

| nhóm k | nút | L thực tế | chẵn? | sau xử lý |
|---|---|---|---|---|
| 1 | `[1]` | 1 | không | `[1]` |
| 2 | `[1,0]` | 2 | có | `[0,1]` |
| 3 | `[6,5]` | **2** (< 3) | có | `[5,6]` |

Kết quả `[1,0,1,5,6]` ✓.

## 5. Cách 1 — Brute Force / baseline: chuyển sang mảng, đảo đoạn, dựng lại

In [4]:
def reverse_groups_array(head):
    v = to_list(head)
    for start, L in groups(len(v)):
        if L % 2 == 0:
            v[start:start + L] = v[start:start + L][::-1]
    return build(v)

for vals, exp in examples:
    got = to_list(reverse_groups_array(build(vals))); print(got, "kỳ vọng", exp); assert got == exp

[5, 6, 2, 3, 9, 1, 4, 8, 3, 7] kỳ vọng [5, 6, 2, 3, 9, 1, 4, 8, 3, 7]
[1, 0, 1, 6] kỳ vọng [1, 0, 1, 6]
[1, 0, 1, 5, 6] kỳ vọng [1, 0, 1, 5, 6]


**Độ phức tạp:** `O(n)` thời gian nhưng `O(n)` bộ nhớ phụ và tạo **nút mới** (không sửa danh sách gốc). Dùng làm chuẩn đối chiếu.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** chỉ độ dài **thực tế** quyết định đảo; với mọi nhóm trừ nhóm cuối, thực tế = `k`.

**Insight 2:** đảo tại chỗ một đoạn `L` nút sau `prev` trong `O(L)`, trả về đuôi mới để làm `prev` cho nhóm sau.

**Insight 3:** tổng công việc = `Σ (đếm L + đảo L) = O(n)`.

**Invariant:** trước khi xử lý nhóm `k`, `prev` là nút cuối của nhóm `k − 1` (đã xử lý xong), và phần trước `prev` đã đúng kết quả.

### 6.1 Minh họa: độ dài thực tế của nhóm cuối với `n = 1..12`

In [5]:
for n in range(1, 13):
    g = groups(n); last = g[-1]
    print(f"n={n:>2}: nhóm cuối bắt đầu ở {last[0]}, dài {last[1]} (dự kiến {len(g)}) -> {'đảo' if last[1] % 2 == 0 else 'giữ'}")

n= 1: nhóm cuối bắt đầu ở 0, dài 1 (dự kiến 1) -> giữ
n= 2: nhóm cuối bắt đầu ở 1, dài 1 (dự kiến 2) -> giữ
n= 3: nhóm cuối bắt đầu ở 1, dài 2 (dự kiến 2) -> đảo
n= 4: nhóm cuối bắt đầu ở 3, dài 1 (dự kiến 3) -> giữ
n= 5: nhóm cuối bắt đầu ở 3, dài 2 (dự kiến 3) -> đảo
n= 6: nhóm cuối bắt đầu ở 3, dài 3 (dự kiến 3) -> giữ
n= 7: nhóm cuối bắt đầu ở 6, dài 1 (dự kiến 4) -> giữ
n= 8: nhóm cuối bắt đầu ở 6, dài 2 (dự kiến 4) -> đảo
n= 9: nhóm cuối bắt đầu ở 6, dài 3 (dự kiến 4) -> giữ
n=10: nhóm cuối bắt đầu ở 6, dài 4 (dự kiến 4) -> đảo
n=11: nhóm cuối bắt đầu ở 10, dài 1 (dự kiến 5) -> giữ
n=12: nhóm cuối bắt đầu ở 10, dài 2 (dự kiến 5) -> đảo


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Đảo tại chỗ từng nhóm (bản nộp)

In [6]:
class Solution:
    def reverseEvenLengthGroups(self, head: Optional[ListNode]) -> Optional[ListNode]:
        dummy = ListNode(0, head)
        prev, k = dummy, 1
        while prev.next:
            L, node = 0, prev.next
            while node and L < k:                     # đếm độ dài thực tế
                L += 1; node = node.next
            if L % 2 == 0:
                first = prev.next; cur = first; rev = None
                for _ in range(L):
                    nxt = cur.next; cur.next = rev; rev = cur; cur = nxt
                prev.next = rev
                first.next = cur
                prev = first
            else:
                for _ in range(L):
                    prev = prev.next
            k += 1
        return dummy.next

print([to_list(Solution().reverseEvenLengthGroups(build(v))) for v, _ in examples])

[[5, 6, 2, 3, 9, 1, 4, 8, 3, 7], [1, 0, 1, 6], [1, 0, 1, 5, 6]]


**Độ phức tạp Cách 2:** mỗi nút được đếm một lần và đảo (hoặc bước qua) một lần ⇒ `O(n)`; `O(1)` bộ nhớ phụ.

### 7.2 Cách 3 — Đổi **giá trị** trong nhóm chẵn (giữ nguyên liên kết)

Thu các nút của nhóm vào list, rồi hoán đổi `val` đối xứng. Đơn giản, ít lỗi con trỏ; bộ nhớ `O(k)` cho nhóm hiện tại.

In [7]:
def reverse_groups_values(head):
    node, k = head, 1
    while node:
        grp = []
        while node and len(grp) < k:
            grp.append(node); node = node.next
        if len(grp) % 2 == 0:
            i, j = 0, len(grp) - 1
            while i < j:
                grp[i].val, grp[j].val = grp[j].val, grp[i].val; i += 1; j -= 1
        k += 1
    return head

print([to_list(reverse_groups_values(build(v))) for v, _ in examples])

[[5, 6, 2, 3, 9, 1, 4, 8, 3, 7], [1, 0, 1, 6], [1, 0, 1, 5, 6]]


Lưu ý: cách này sửa **giá trị**, không sửa liên kết — nếu đề yêu cầu "đảo nút" theo nghĩa liên kết (nút mang dữ liệu khác), phải dùng Cách 2.

### 7.3 Cách 4 — Tách nhóm thành danh sách con, đảo, nối lại

In [8]:
def reverse_groups_split(head):
    dummy = ListNode(); tail = dummy; node, k = head, 1
    while node:
        g_head = node; L = 1
        while L < k and node.next:
            node = node.next; L += 1
        nxt = node.next; node.next = None                 # cắt nhóm
        if L % 2 == 0:
            prev = None; cur = g_head
            while cur: cur.next, prev, cur = prev, cur, cur.next
            tail.next = prev; tail = g_head
        else:
            tail.next = g_head; tail = node
        node = nxt; k += 1
    return dummy.next

print([to_list(reverse_groups_split(build(v))) for v, _ in examples])

[[5, 6, 2, 3, 9, 1, 4, 8, 3, 7], [1, 0, 1, 6], [1, 0, 1, 5, 6]]


## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 1**:

In [9]:
def trace(vals):
    dummy = ListNode(0, build(vals)); prev, k = dummy, 1
    while prev.next:
        L, node = 0, prev.next
        while node and L < k: L += 1; node = node.next
        seg = []; t = prev.next
        for _ in range(L): seg.append(t.val); t = t.next
        if L % 2 == 0:
            first = prev.next; cur = first; rev = None
            for _ in range(L): nxt = cur.next; cur.next = rev; rev = cur; cur = nxt
            prev.next = rev; first.next = cur; prev = first; act = "đảo"
        else:
            for _ in range(L): prev = prev.next
            act = "giữ"
        print(f"nhóm {k}: {seg} (L = {L}) -> {act:<4} danh sách: {to_list(dummy.next)}")
        k += 1
    return to_list(dummy.next)

assert trace([5, 2, 6, 3, 9, 1, 7, 3, 8, 4]) == [5, 6, 2, 3, 9, 1, 4, 8, 3, 7]

nhóm 1: [5] (L = 1) -> giữ  danh sách: [5, 2, 6, 3, 9, 1, 7, 3, 8, 4]
nhóm 2: [2, 6] (L = 2) -> đảo  danh sách: [5, 6, 2, 3, 9, 1, 7, 3, 8, 4]
nhóm 3: [3, 9, 1] (L = 3) -> giữ  danh sách: [5, 6, 2, 3, 9, 1, 7, 3, 8, 4]
nhóm 4: [7, 3, 8, 4] (L = 4) -> đảo  danh sách: [5, 6, 2, 3, 9, 1, 4, 8, 3, 7]


| nhóm | nút | L | hành động | danh sách |
|---|---|---|---|---|
| 1 | `[5]` | 1 | giữ | `5,2,6,3,9,1,7,3,8,4` |
| 2 | `[2,6]` | 2 | đảo | `5,6,2,3,9,1,7,3,8,4` |
| 3 | `[3,9,1]` | 3 | giữ | |
| 4 | `[7,3,8,4]` | 4 | đảo | `5,6,2,3,9,1,4,8,3,7` |

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — đảo tại chỗ.

In [10]:
class Solution:
    def reverseEvenLengthGroups(self, head: Optional[ListNode]) -> Optional[ListNode]:
        dummy = ListNode(0, head)
        prev, k = dummy, 1
        while prev.next:
            L, node = 0, prev.next
            while node and L < k:
                L += 1
                node = node.next
            if L % 2 == 0:
                first = prev.next
                cur, rev = first, None
                for _ in range(L):
                    nxt = cur.next
                    cur.next = rev
                    rev = cur
                    cur = nxt
                prev.next = rev
                first.next = cur
                prev = first
            else:
                for _ in range(L):
                    prev = prev.next
            k += 1
        return dummy.next

for vals, exp in examples:
    got = to_list(Solution().reverseEvenLengthGroups(build(vals))); print(got, exp); assert got == exp

[5, 6, 2, 3, 9, 1, 4, 8, 3, 7] [5, 6, 2, 3, 9, 1, 4, 8, 3, 7]
[1, 0, 1, 6] [1, 0, 1, 6]
[1, 0, 1, 5, 6] [1, 0, 1, 5, 6]


## 10. Vì sao thuật toán đúng

**Invariant.** Trước vòng lặp với nhóm `k`, `prev` là nút cuối của phần đã xử lý (nhóm `1..k−1` theo kết quả đúng) và `prev.next` là nút đầu tiên của nhóm `k` trong thứ tự **gốc** (vì ta chỉ sửa liên kết bên trong các nhóm đã xử lý và nối lại đúng phần sau).

**Một bước.** Vòng đếm cho `L = min(k, số nút còn lại)` — đúng độ dài thực tế của nhóm `k`. Nếu `L` chẵn: đảo `L` nút đầu tiên sau `prev`; nối `prev.next` vào nút cuối cũ, nút đầu cũ vào phần sau ⇒ nhóm `k` đúng kết quả; nút đầu cũ trở thành cuối phần đã xử lý ⇒ `prev = first`. Nếu `L` lẻ: tiến `prev` qua `L` nút. Invariant giữ.

**Kết thúc.** Khi `prev.next` là `None`, mọi nút đã được xử lý. Mỗi nút thuộc đúng một nhóm theo định nghĩa. ∎

### 10.1 Kiểm chứng thực nghiệm (kể cả định danh nút)

In [11]:
rng = random.Random(1010)
for _ in range(1000):
    vals = [rng.randint(0, 9) for _ in range(rng.randint(1, 40))]
    ref = to_list(reverse_groups_array(build(vals)))
    for fn in (Solution().reverseEvenLengthGroups, reverse_groups_values, reverse_groups_split):
        h = build(vals); assert to_list(fn(h)) == ref
    h = build(vals); ids = []; c = h
    while c: ids.append(id(c)); c = c.next
    out = Solution().reverseEvenLengthGroups(h); ids2 = []; c = out
    while c: ids2.append(id(c)); c = c.next
    assert sorted(ids) == sorted(ids2)
print("4 cách khớp nhau trên 1000 danh sách; bản nộp giữ nguyên tập nút")

4 cách khớp nhau trên 1000 danh sách; bản nộp giữ nguyên tập nút


## 11. Phân tích Time / Space Complexity

| Cách | Time | Space phụ |
|---|---|---|
| 1. Mảng | `O(n)` | `O(n)` |
| 2. Đảo tại chỗ | `O(n)` | `O(1)` |
| 3. Đổi giá trị | `O(n)` | `O(√n)` (nhóm lớn nhất) |
| 4. Tách–đảo–nối | `O(n)` | `O(1)` |

In [12]:
big = list(range(10 ** 5))
for name, f in [("array", reverse_groups_array), ("inplace", Solution().reverseEvenLengthGroups), ("values", reverse_groups_values), ("split", reverse_groups_split)]:
    h = build(big); t = time.perf_counter(); r = to_list(f(h)); print(f"{name:<8}: {(time.perf_counter() - t) * 1000:.0f} ms, 5 phần tử cuối {r[-5:]}")

array   : 65 ms, 5 phần tử cuối [99995, 99996, 99997, 99998, 99999]


inplace : 9 ms, 5 phần tử cuối [99995, 99996, 99997, 99998, 99999]

values  : 11 ms, 5 phần tử cuối [99995, 99996, 99997, 99998, 99999]


split   : 9 ms, 5 phần tử cuối [99995, 99996, 99997, 99998, 99999]


## 12. Bộ test

In [13]:
ALL = {"array": reverse_groups_array, "inplace": Solution().reverseEvenLengthGroups, "values": reverse_groups_values, "split": reverse_groups_split}
def run_all(vals):
    res = {k: to_list(f(build(vals))) for k, f in ALL.items()}
    vs = list(res.values()); assert all(v == vs[0] for v in vs), (vals, res)
    return vs[0]

for vals, exp in examples:
    assert run_all(vals) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — kiểm bằng bản mảng trong run_all
edge = [([1], [1]), ([1, 2], [1, 2]), ([1, 2, 3], [1, 3, 2]), ([1, 2, 3, 4, 5], [1, 3, 2, 5, 4]), ([1, 2, 3, 4, 5, 6], [1, 3, 2, 4, 5, 6])]
for vals, exp in edge:
    assert run_all(vals) == exp, (vals, run_all(vals))
rng = random.Random(20740)
for _ in range(500):
    run_all([rng.randint(0, 100) for _ in range(rng.randint(1, 60))])
print("Ví dụ, 5 edge case, 500 danh sách ngẫu nhiên: OK")

Ví dụ, 5 edge case, 500 danh sách ngẫu nhiên: OK


### 12.1 Kiểm tra tính chất

- Áp dụng thao tác **hai lần** cho lại danh sách gốc (đảo nhóm chẵn là phép tự nghịch đảo vì ranh giới nhóm chỉ phụ thuộc vị trí).
- Đa tập giá trị giữ nguyên; nút đầu không đổi.

In [14]:
rng = random.Random(121)
f = Solution().reverseEvenLengthGroups
for _ in range(500):
    v = [rng.randint(0, 9) for _ in range(rng.randint(1, 50))]
    once = to_list(f(build(v))); twice = to_list(f(build(once)))
    assert twice == v and sorted(once) == sorted(v) and once[0] == v[0]
print("Tự nghịch đảo, giữ đa tập, giữ nút đầu: đúng trên 500 danh sách")

Tự nghịch đảo, giữ đa tập, giữ nút đầu: đúng trên 500 danh sách


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quyết định đảo theo độ dài **dự kiến** `k` thay vì thực tế

In [15]:
def wrong_expected_len(head):
    v = to_list(head); start, k = 0, 1
    while start < len(v):
        L = min(k, len(v) - start)
        if k % 2 == 0:                                            # SAI: dùng k
            v[start:start + L] = v[start:start + L][::-1]
        start += L; k += 1
    return build(v)

for vals, exp in examples:
    print("sai :", to_list(wrong_expected_len(build(vals))), "| đúng:", exp)

sai : [5, 6, 2, 3, 9, 1, 4, 8, 3, 7] | đúng: [5, 6, 2, 3, 9, 1, 4, 8, 3, 7]
sai : [1, 0, 1, 6] | đúng: [1, 0, 1, 6]
sai : [1, 0, 1, 6, 5] | đúng: [1, 0, 1, 5, 6]


**Vì sao sai:** nhóm cuối có thể ngắn hơn: Ví dụ 2 nhóm cuối `[6]` (dự kiến 3, thực tế 1 — giữ), Ví dụ 3 nhóm cuối `[6,5]` (dự kiến 3 — bản sai **không** đảo; thực tế 2 — phải đảo). **Sửa:** đếm độ dài thực tế.

### Lỗi 2 — Sau khi đảo, đặt `prev` vào **đầu mới** thay vì đuôi mới

In [16]:
def wrong_prev(head):
    dummy = ListNode(0, head); prev, k = dummy, 1
    while prev.next:
        L, node = 0, prev.next
        while node and L < k: L += 1; node = node.next
        if L % 2 == 0:
            first = prev.next; cur = first; rev = None
            for _ in range(L): nxt = cur.next; cur.next = rev; rev = cur; cur = nxt
            prev.next = rev; first.next = cur
            prev = rev                                            # SAI: rev là đầu mới, không phải đuôi
        else:
            for _ in range(L): prev = prev.next
        k += 1
        if k > 1000: break
    return dummy.next

def safe(h):
    try: return to_list(h, limit=100)
    except RuntimeError as e: return f"lỗi: {e}"
for vals, exp in examples:
    print("sai :", safe(wrong_prev(build(vals))), "| đúng:", exp)

sai : [5, 6, 2, 3, 9, 8, 4, 1, 7, 3] | đúng: [5, 6, 2, 3, 9, 1, 4, 8, 3, 7]
sai : [1, 0, 6, 1] | đúng: [1, 0, 1, 6]
sai : [1, 0, 1, 6, 5] | đúng: [1, 0, 1, 5, 6]


**Vì sao sai:** sau khi đảo, nút **đầu** cũ (`first`) mới là nút cuối của nhóm; đặt `prev = rev` khiến nhóm kế tiếp bắt đầu sai chỗ (đè lên nhóm vừa đảo). **Sửa:** `prev = first`.

### Lỗi 3 — Quên nối đuôi mới với phần sau (`first.next = cur`)

In [17]:
def wrong_no_reconnect(head):
    dummy = ListNode(0, head); prev, k = dummy, 1
    while prev.next:
        L, node = 0, prev.next
        while node and L < k: L += 1; node = node.next
        if L % 2 == 0:
            first = prev.next; cur = first; rev = None
            for _ in range(L): nxt = cur.next; cur.next = rev; rev = cur; cur = nxt
            prev.next = rev; prev = first                         # SAI: first.next vẫn là None
        else:
            for _ in range(L): prev = prev.next
        k += 1
    return dummy.next

for vals, exp in examples:
    print("sai :", to_list(wrong_no_reconnect(build(vals))), "| đúng:", exp)

sai : [5, 6, 2] | đúng: [5, 6, 2, 3, 9, 1, 4, 8, 3, 7]
sai : [1, 0, 1] | đúng: [1, 0, 1, 6]
sai : [1, 0, 1] | đúng: [1, 0, 1, 5, 6]


**Vì sao sai:** sau khi đảo, `first.next = None` (nó từng là nút đầu tiên được đảo) ⇒ danh sách bị **cắt** sau nhóm chẵn đầu tiên. **Sửa:** `first.next = cur`.

### Lỗi 4 — Đánh số nhóm bắt đầu từ 0 (hoặc tăng `k` sai chỗ)

In [18]:
def wrong_k0(head):
    v = to_list(head); start, k = 0, 0
    while start < len(v):
        L = min(max(k, 1), len(v) - start)                        # SAI: nhóm đầu "0" rồi bị ép thành 1
        if L % 2 == 0: v[start:start + L] = v[start:start + L][::-1]
        start += L; k += 1
    return build(v)

for vals, exp in examples:
    print("sai :", to_list(wrong_k0(build(vals))), "| đúng:", exp)

sai : [5, 2, 3, 6, 9, 1, 7, 3, 8, 4] | đúng: [5, 6, 2, 3, 9, 1, 4, 8, 3, 7]
sai : [1, 1, 6, 0] | đúng: [1, 0, 1, 6]
sai : [1, 1, 6, 0, 5] | đúng: [1, 0, 1, 5, 6]


**Vì sao sai:** độ dài nhóm phải là `1, 2, 3, …`; bắt đầu `k = 0` tạo dãy `1, 1, 2, 3, …` lệch ranh giới. **Sửa:** `k = 1` và tăng sau mỗi nhóm.

### Lỗi 5 — Đảo cả nhóm lẻ (hiểu ngược điều kiện)

In [19]:
def wrong_parity(head):
    v = to_list(head)
    for start, L in groups(len(v)):
        if L % 2 == 1: v[start:start + L] = v[start:start + L][::-1]      # SAI
    return build(v)

for vals, exp in examples:
    print("sai :", to_list(wrong_parity(build(vals))), "| đúng:", exp)

sai : [5, 2, 6, 1, 9, 3, 7, 3, 8, 4] | đúng: [5, 6, 2, 3, 9, 1, 4, 8, 3, 7]
sai : [1, 1, 0, 6] | đúng: [1, 0, 1, 6]
sai : [1, 1, 0, 6, 5] | đúng: [1, 0, 1, 5, 6]


**Vì sao sai:** đề: đảo nhóm có độ dài **chẵn**. **Sửa:** `L % 2 == 0`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đảo một đoạn trong danh sách liên kết: lưu `prev` (trước đoạn), `first` (đầu đoạn, sẽ thành đuôi), đảo `L` nút, nối `prev.next = đầu mới`, `first.next = phần sau`.
- Quyết định dựa trên **đếm thực tế** — luôn đếm trước khi đảo.
- Dummy node cho mọi thao tác có thể đổi đầu.
- Liên quan: **25 Reverse Nodes in k-Group**, **92 Reverse Linked List II**.

## 15. Biến thể của bài

1. **25 Reverse Nodes in k-Group** (Hard): nhóm cố định `k`, nhóm cuối thiếu thì giữ.
2. **92 Reverse Linked List II**: đảo đoạn `[left, right]`.
3. **Đảo nhóm có độ dài lẻ** / **đảo nhóm có chỉ số chẵn**.
4. **Nhóm theo dãy Fibonacci** hoặc độ dài tùy ý — tham số hóa hàm sinh độ dài.
5. **24 Swap Nodes in Pairs**: nhóm cố định 2.

### 15.1 Code biến thể: 25 Reverse Nodes in k-Group

In [20]:
def reverse_k_group(head, k):
    dummy = ListNode(0, head); prev = dummy
    while True:
        node, L = prev.next, 0
        while node and L < k: node = node.next; L += 1
        if L < k: return dummy.next
        first = prev.next; cur = first; rev = None
        for _ in range(k): nxt = cur.next; cur.next = rev; rev = cur; cur = nxt
        prev.next = rev; first.next = cur; prev = first

rng = random.Random(25)
for _ in range(300):
    v = [rng.randint(0, 9) for _ in range(rng.randint(1, 20))]; k = rng.randint(1, 6)
    exp = []
    for i in range(0, len(v), k):
        chunk = v[i:i + k]; exp += chunk[::-1] if len(chunk) == k else chunk
    assert to_list(reverse_k_group(build(v), k)) == exp
print("25 khớp bản mảng;", to_list(reverse_k_group(build([1, 2, 3, 4, 5]), 2)))

25 khớp bản mảng; [2, 1, 4, 3, 5]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mảng | Đảo lát cắt | `O(n)` | `O(n)` | Kiểm chứng |
| 2. Đảo tại chỗ | Con trỏ | `O(n)` | `O(1)` | **Bản nộp** |
| 3. Đổi giá trị | Hoán đổi `val` | `O(n)` | `O(√n)` | Ít lỗi con trỏ |
| 4. Tách–đảo–nối | Danh sách con | `O(n)` | `O(1)` | Tư duy theo khối |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `[1,2,3]`; (b) `[1,2,3,4,5]`; (c) `[1,2,3,4,5,6]`; (d) `[1,2]`.

**Bài 2 (tính tay):** Chia nhóm cho `n = 14` và cho biết những nhóm nào bị đảo.

**Bài 3 (điền TODO):** Hoàn thiện `group_lengths(n)` trả danh sách độ dài thực tế các nhóm.

**Bài 4 (tự viết lại):** Viết `reverse_odd_length_groups(head)` — đảo các nhóm có độ dài **lẻ**.

**Bài 5 (mini challenge):** Viết `reverse_by_lengths(head, lengths_fn)` tổng quát: độ dài nhóm thứ `k` là `lengths_fn(k)` (ví dụ Fibonacci), đảo nhóm có độ dài thực tế chẵn; đối chiếu bản mảng với `lengths_fn = fib`.

In [21]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def group_lengths(n):
    out, start, k = [], 0, 1
    while start < n:
        # TODO: L = min(k, n - start); out.append(L); start += L; k += 1
        pass
    return out

In [22]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def reverse_odd_length_groups(head):
    # TODO: như Cách 2 nhưng đảo khi L % 2 == 1
    pass

def reverse_by_lengths(head, lengths_fn):
    # TODO: như Cách 2 nhưng k-th group expected length = lengths_fn(k)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: chia nhóm. Mức 2: đảo nhóm chẵn. Mức 3: (a) `[1,3,2]`; (b) `[1,3,2,5,4]` (nhóm cuối `[4,5]` dài 2); (c) `[1,3,2,4,5,6]`; (d) `[1,2]` (nhóm cuối `[2]` dài 1).

**Bài 2.** Mức 1: nhóm dài `1,2,3,4,4` (nhóm 5 chỉ còn 4 nút). Mức 2: đảo nhóm 2, 4 và nhóm cuối (dài 4). Mức 3: dùng `groups(14)`.

**Bài 3.** Mức 1: biến `start`, `k`. Mức 2: `min(k, n − start)`. Mức 3: tăng `k` mỗi nhóm.

**Bài 4.** Mức 1: đổi điều kiện. Mức 2: nhóm 1 (dài 1) bị "đảo" nhưng không đổi. Mức 3: đối chiếu bản mảng.

**Bài 5.** Mức 1: thay `k` bằng `lengths_fn(k)`. Mức 2: đếm thực tế. Mức 3: Fibonacci `1, 1, 2, 3, 5, …`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2, 3
f = Solution().reverseEvenLengthGroups
b1 = [to_list(f(build(v))) for v in ([1, 2, 3], [1, 2, 3, 4, 5], [1, 2, 3, 4, 5, 6], [1, 2])]
print("Bài 1:", b1); assert b1 == [[1, 3, 2], [1, 3, 2, 5, 4], [1, 3, 2, 4, 5, 6], [1, 2]]
print("Bài 2:", groups(14), "-> đảo các nhóm", [i + 1 for i, (_, L) in enumerate(groups(14)) if L % 2 == 0])
def group_lengths(n):
    out, start, k = [], 0, 1
    while start < n:
        L = min(k, n - start); out.append(L); start += L; k += 1
    return out
assert all(group_lengths(n) == [L for _, L in groups(n)] for n in range(1, 50))
print("Bài 3: OK")

Bài 1: [[1, 3, 2], [1, 3, 2, 5, 4], [1, 3, 2, 4, 5, 6], [1, 2]]
Bài 2: [(0, 1), (1, 2), (3, 3), (6, 4), (10, 4)] -> đảo các nhóm [2, 4, 5]
Bài 3: OK


In [24]:
# Bài 4, 5
def reverse_by_lengths(head, lengths_fn, want_even=True):
    dummy = ListNode(0, head); prev, k = dummy, 1
    while prev.next:
        target = lengths_fn(k); L, node = 0, prev.next
        while node and L < target: L += 1; node = node.next
        if (L % 2 == 0) == want_even:
            first = prev.next; cur = first; rev = None
            for _ in range(L): nxt = cur.next; cur.next = rev; rev = cur; cur = nxt
            prev.next = rev; first.next = cur; prev = first
        else:
            for _ in range(L): prev = prev.next
        k += 1
    return dummy.next

def reverse_odd_length_groups(head):
    return reverse_by_lengths(head, lambda k: k, want_even=False)

def fib(k):
    a, b = 1, 1
    for _ in range(k - 1): a, b = b, a + b
    return a

def array_by_lengths(v, lengths_fn, want_even=True):
    v = v[:]; start, k = 0, 1
    while start < len(v):
        L = min(lengths_fn(k), len(v) - start)
        if (L % 2 == 0) == want_even: v[start:start + L] = v[start:start + L][::-1]
        start += L; k += 1
    return v

rng = random.Random(5)
for _ in range(300):
    v = [rng.randint(0, 9) for _ in range(rng.randint(1, 40))]
    assert to_list(reverse_odd_length_groups(build(v))) == array_by_lengths(v, lambda k: k, False)
    assert to_list(reverse_by_lengths(build(v), fib)) == array_by_lengths(v, fib)
    assert to_list(reverse_by_lengths(build(v), lambda k: k)) == to_list(f(build(v)))
print("Bài 4, 5: OK |", to_list(reverse_odd_length_groups(build([1, 2, 3, 4, 5, 6]))), to_list(reverse_by_lengths(build(list(range(1, 13))), fib)))

Bài 4, 5: OK | [1, 2, 3, 6, 5, 4] [1, 2, 4, 3, 5, 6, 7, 8, 9, 10, 11, 12]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
dummy = ListNode(0, head); prev, k = dummy, 1
while prev.next:
    L = đếm tối đa k nút sau prev
    if L % 2 == 0:
        first = prev.next; cur, rev = first, None
        for _ in range(L): cur.next, rev, cur = rev, cur, cur.next
        prev.next = rev; first.next = cur; prev = first
    else:
        tiến prev qua L nút
    k += 1
return dummy.next
```

**Checklist:**
- [ ] Dùng độ dài **thực tế** (nhóm cuối có thể thiếu)?
- [ ] Nối `first.next = cur` sau khi đảo?
- [ ] `prev = first` (đuôi mới) sau khi đảo?
- [ ] Nhóm bắt đầu từ `k = 1`?
- [ ] Chỉ đảo nhóm **chẵn**?